# 04 Model Monitoring Simulation

Simulation of data drift, missing-value increase, prediction drift, threshold sensitivity, and retraining triggers.

In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()))

import numpy as np
import pandas as pd
from src.config import SAMPLE_DIR
from src.monitoring import population_stability_index, simulate_monitoring_snapshot
from src.evaluate import threshold_table

In [ ]:
try:
    scored = pd.read_csv(SAMPLE_DIR / 'scored_components_sample.csv')
except FileNotFoundError:
    rng = np.random.default_rng(42)
    scored = pd.DataFrame({'risk_score': rng.beta(1.4, 14, 5000), 'Response': rng.binomial(1, 0.006, 5000)})

monitoring_snapshot = simulate_monitoring_snapshot(scored)
monitoring_snapshot

## Data Drift And Missingness Increase

In [ ]:
rng = np.random.default_rng(42)
baseline_scores = scored['risk_score']
current_scores = np.clip(baseline_scores * 1.35 + rng.normal(0, 0.03, len(scored)), 0, 1)
psi = population_stability_index(baseline_scores, pd.Series(current_scores))
psi

## Performance Degradation And Threshold Sensitivity

In [ ]:
if 'Response' in scored and scored['Response'].notna().any():
    y = scored['Response'].fillna(0).astype(int)
else:
    y = rng.binomial(1, 0.006, len(scored))
threshold_table(y, current_scores, thresholds=[0.05, 0.10, 0.15, 0.20, 0.30, 0.50])

## Alert Logic

Amber alerts indicate required quality review. Red alerts indicate governance review, possible threshold adjustment, or retraining workflow initiation. Retraining should not be automatic unless approved by the model governance process.